# Task 5: Mental Health Support Chatbot (Fine-Tuned)

Objective:

Build a basic chatbot that provides supportive and empathetic responses for stress, anxiety,
and emotional wellness.

In [1]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [2]:
import transformers
import datasets
import accelerate

print(transformers.__version__)
print(datasets.__version__)
print(accelerate.__version__)

5.12.0
4.0.0
1.14.0


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

print("Tokenizer Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer Loaded Successfully


In [4]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

print("Model Loaded Successfully")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Loaded Successfully


In [5]:
from datasets import Dataset

data = {
    "text": [
        "User: I feel stressed about exams. Bot: It's understandable to feel stressed. Try taking one step at a time and remember to take breaks.",
        "User: I am anxious about my future. Bot: Many people feel uncertain about the future. Focus on what you can control today.",
        "User: I feel lonely. Bot: I'm sorry you're feeling lonely. Reaching out to someone you trust may help.",
        "User: I am overwhelmed with work. Bot: That sounds difficult. Consider prioritizing tasks and giving yourself time to rest.",
        "User: I feel sad and unmotivated. Bot: It's okay to have difficult days. Be kind to yourself and take small steps forward.",
        "User: Nobody understands me. Bot: Feeling misunderstood can be painful. Your feelings are valid and deserve attention.",
        "User: I keep worrying about everything. Bot: Constant worry can be exhausting. Try focusing on one concern at a time.",
        "User: I failed my test. Bot: I'm sorry that happened. One setback doesn't define your abilities or future success."
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 8
})


In [6]:
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_data = dataset.map(tokenize)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [7]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="epoch"
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data
)

In [9]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,9.519487
2,6.302206
3,2.820601
4,1.332367
5,1.048491
6,1.007842
7,0.915292
8,1.009042
9,0.929978
10,0.828258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=1.6510742217302323, metrics={'train_runtime': 35.1653, 'train_samples_per_second': 1.137, 'train_steps_per_second': 0.569, 'total_flos': 1306483752960.0, 'train_loss': 1.6510742217302323, 'epoch': 5.0})

In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
print("Tokenizer Loaded Successfully")

Tokenizer Loaded Successfully


In [11]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
print("Model Loaded Successfully")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model Loaded Successfully


In [23]:
from datasets import Dataset

data = {
    "text": [
        "User: I feel stressed about exams. Bot: It's understandable to feel stressed. Try taking one step at a time and remember to take breaks.",

        "User: I am anxious about my future. Bot: Many people feel uncertain about the future. Focus on what you can control today.",

        "User: I feel lonely. Bot: I'm sorry you're feeling lonely. Reaching out to someone you trust may help.",

        "User: I am overwhelmed with work. Bot: That sounds difficult. Consider prioritizing tasks and giving yourself time to rest.",

        "User: I feel sad and unmotivated. Bot: It's okay to have difficult days. Be kind to yourself and take small steps forward.",

        "User: Nobody understands me. Bot: Feeling misunderstood can be painful. Your feelings are valid and deserve attention.",

        "User: I keep worrying about everything. Bot: Constant worry can be exhausting. Try focusing on one concern at a time.",

        "User: I failed my test. Bot: I'm sorry that happened. One setback doesn't define your abilities or future success.",

        "User: I feel anxious before presentations. Bot: It's normal to feel nervous before speaking. Preparation and practice can help build confidence.",

        "User: I am worried about my grades. Bot: Academic pressure can be overwhelming. Focus on steady progress rather than perfection.",

        "User: I feel exhausted all the time. Bot: It sounds like you've been carrying a lot lately. Make sure you're giving yourself time to rest and recover.",

        "User: I don't think I am good enough. Bot: Self-doubt can be difficult. Remember that your worth is not defined by one challenge or mistake.",

        "User: I had a bad day. Bot: I'm sorry your day was difficult. Tomorrow is a new opportunity and today doesn't define everything.",

        "User: I feel overwhelmed by responsibilities. Bot: Breaking large tasks into smaller steps can make things feel more manageable."
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 14
})


In [24]:
tokenizer.pad_token = tokenizer.eos_token

In [25]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_data = dataset.map(tokenize)

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

In [26]:
print(tokenized_data[0])

{'text': "User: I feel stressed about exams. Bot: It's understandable to feel stressed. Try taking one step at a time and remember to take breaks.", 'input_ids': [12982, 25, 314, 1254, 15033, 546, 26420, 13, 18579, 25, 632, 338, 21977, 284, 1254, 15033, 13, 9993, 2263, 530, 2239, 379, 257, 640, 290, 3505, 284, 1011, 9457, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 

In [27]:
model.config.pad_token_id = tokenizer.eos_token_id

In [28]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data
)

trainer.train()

Step,Training Loss
1,0.201881
2,0.123455
3,0.206662
4,0.205403
5,0.152644
6,0.155907
7,0.350795
8,0.154937
9,0.191297
10,0.123132


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=35, training_loss=0.15027400553226472, metrics={'train_runtime': 8.7247, 'train_samples_per_second': 8.023, 'train_steps_per_second': 4.012, 'total_flos': 2286346567680.0, 'train_loss': 0.15027400553226472, 'epoch': 5.0})

In [31]:
trainer.train()

Step,Training Loss
1,0.079486
2,0.063577
3,0.121400
4,0.087729
5,0.102837
6,0.139636
7,0.209467
8,0.097860
9,0.128481
10,0.079914


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=35, training_loss=0.09679005231176104, metrics={'train_runtime': 25.405, 'train_samples_per_second': 2.755, 'train_steps_per_second': 1.378, 'total_flos': 2286346567680.0, 'train_loss': 0.09679005231176104, 'epoch': 5.0})

In [32]:
trainer.save_model("./mental_health_model")
tokenizer.save_pretrained("./mental_health_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mental_health_model/tokenizer_config.json',
 './mental_health_model/tokenizer.json')

In [33]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="./mental_health_model",
    tokenizer="./mental_health_model"
)

questions = [
    "I feel stressed about exams.",
    "I am anxious about my future.",
    "I feel lonely.",
    "I failed my test.",
    "I feel overwhelmed."
]

for q in questions:

    prompt = f"""
User: {q}
Bot:
"""

    response = chatbot(
        prompt,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.2
    )

    print("="*60)
    print(response[0]["generated_text"])
    print()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'repetition_penalty', 'top_k', 'do_sample', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to


User: I feel stressed about exams.
Bot:
Unfortunately, the exam schedule doesn't define everything you need to do today. Preparation and practice can help build confidence in yourself before taking a step back from difficult tasks into great steps forward.


User: I am anxious about my future.
Bot:




[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



User: I feel lonely.
Bot:
I'm sorry that happened and you're feeling sad today. Try focusing on one concern at a time and remember to take breaks before worrying too much about everything


User: I failed my test.
Bot:
I'm sorry that happened and apologize to you at the hands of someone else.


User: I feel overwhelmed.
Bot:
I'm sorry that happened, but it's okay to have difficult days. Be kind and take small steps forward in your daily lives.

